# app17-6 — treinamento do modelo binário (normal × anomalia)

Lê as janelas gravadas no InfluxDB pelo fluxo `FluxoNodeRED-Binary` e gera o
`modelo_vibracao_binaria.pkl`, que o app seguinte de inferência usando IMU carrega
para responder no MQTT.

A consulta é em **SQL**, pelo cliente do InfluxDB 3.

In [ ]:
# Versoes fixas: o .pkl gerado aqui e carregado pela API do app seguinte com
# EXATAMENTE estas versoes (api/requirements.txt). As duas que precisam
# casar de verdade sao numpy e scikit-learn: e nelas que o formato do
# .pkl se apoia. Se o Colab pedir "Restart session" depois desta celula,
# reinicie e rode de novo.
#
# influxdb3-python le por SQL (Arrow Flight); pyarrow vem junto e e quem
# converte o resultado em DataFrame.
!pip install -q influxdb3-python pyarrow matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "joblib==1.5.3"

## 1) Conectar no InfluxDB Cloud

Preencha com os seus valores — os mesmos do nó InfluxDB do Node-RED.

O `bucket` do Node-RED é o `database` aqui. A **organização não entra**: ela é usada na
*escrita* (Node-RED); para ler por SQL bastam host, token e database.

In [ ]:
from influxdb_client_3 import InfluxDBClient3
import pandas as pd

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"
INFLUX_BUCKET = "IoTSensores"          # o "database" no InfluxDB 3
MEASUREMENT   = "vibracao_binario"

FEATURES = ["mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag"]
CLASSES  = ["ligado_normal", "ligado_anomalia"]

client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)

# Confere a conexao e lista as tabelas do database.
display(client.query("SHOW TABLES", language="sql").to_pandas())

### Que colunas o fluxo gravou

`label` e `device` foram gravados como **tags**; as features e o `ts_epoch_ms`, como
**fields**. No SQL isso aparece direto como colunas — tag volta texto, field volta
número. É a diferença prática para o Flux, que devolvia uma linha por field e exigia um
`pivot()` para remontar a janela.

Repare que o fluxo grava `rms_ax`, `rms_ay` e `rms_az` além das sete features usadas
aqui. O `SELECT` abaixo pede só as sete: `rms² = mean² + std²`, então as três por eixo
não trazem informação nova sobre o que `mean_*` e `std_*` já dizem.

In [ ]:
display(client.query(f'SHOW COLUMNS FROM "{MEASUREMENT}"', language="sql").to_pandas())

## 2) Ler as janelas

`SELECT` das colunas que interessam. Sem `pivot`, sem `keep`, sem `range` — o recorte de
tempo é um `WHERE` comum.

In [ ]:
colunas = ", ".join(f'"{f}"' for f in FEATURES)
sql = f'''
SELECT time, "label", {colunas}
FROM "{MEASUREMENT}"
WHERE time >= now() - INTERVAL '30 days'
ORDER BY time
'''

df = client.query(query=sql, language="sql").to_pandas()
client.close()

df = (df.query("label in @CLASSES")
        .dropna(subset=FEATURES)
        .sort_values("time")
        .reset_index(drop=True))

print(f"{len(df)} janelas")
print(df.groupby("label").size().to_string())

## 3) Split cronológico por classe (70/30)

In [ ]:
treino_idx, teste_idx = [], []
for _, g in df.groupby("label"):
    corte = int(len(g) * 0.7)
    treino_idx += list(g.index[:corte])
    teste_idx  += list(g.index[corte:])

treino, teste = df.loc[treino_idx], df.loc[teste_idx]
print(f"treino: {len(treino)} janelas | teste: {len(teste)} janelas")

## 4) Treinar — StandardScaler + rede neural, num Pipeline só

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42),
)

# y em TEXTO: o predict devolve o nome da classe, e a API nao precisa de mapa.
modelo.fit(treino[FEATURES], treino["label"])
print("classes:", list(modelo.classes_))

## 5) Métricas e matriz de confusão

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = modelo.predict(teste[FEATURES])

print("acuracia:", round(accuracy_score(teste["label"], y_pred), 3))
print(classification_report(teste["label"], y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(teste["label"], y_pred,
                                        labels=list(modelo.classes_), xticks_rotation=45)
plt.tight_layout(); plt.show()

## 6) Importância das features (permutation, no conjunto de teste)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(modelo, teste[FEATURES], teste["label"],
                              n_repeats=20, random_state=42)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

imp.plot.barh(color="tab:orange")
plt.title("Permutation importance (teste)")
plt.tight_layout(); plt.show()

print(imp.sort_values(ascending=False).round(3).to_string())

## 7) Salvar o `.pkl`

In [ ]:
import joblib

ARQUIVO = "modelo_vibracao_binaria.pkl"
joblib.dump(modelo, ARQUIVO)

# Confere recarregando, que e exatamente o que a API faz na inicializacao.
recarregado = joblib.load(ARQUIVO)
print(recarregado.predict(teste[FEATURES].head(3)))

# As versoes deste runtime. Precisam ser as mesmas do api/requirements.txt --
# e por isso que a primeira celula as pina.
from importlib.metadata import version
for pacote in ("numpy", "pandas", "scikit-learn", "joblib"):
    print(f"{pacote}=={version(pacote)}")

try:
    from google.colab import files
    files.download(ARQUIVO)
except Exception:
    print(f"{ARQUIVO} gerado na pasta atual.")

Copie o `.pkl` para a pasta `api/` do app seguinte de inferência usando IMU,
substituindo o sintético. Não é preciso mexer no `requirements.txt`: as versões são as
mesmas pinadas na primeira célula — a linha impressa acima é só a conferência.